# PoliMillionaire News and Maths Notebook

TEAM HOMELANDER

Student ID : 11034239 - Karuniaa Jacob Selvi - karuniaa.jacob@mail.polimi.it

Student ID : 11033318 - Balasakthi Shanmugaraja -
 balasakthi.shanmuagaraja@mail.polimi.it

Student ID : 11047334 - Hariharan Murugan Marichamy -
hariharan.murugan@mail.polimi.it

Model Used - Qwen2.5-7B-Instruct

Video Link - <https://youtu.be/MUR7eXemTlw>

This notebook focuses on two PoliMillionaire categories: **News** and **Maths**.

We used the open-weight model **Qwen2.5-7B-Instruct**, loaded locally in Google Colab using the `transformers` library. The model runs locally on the Colab GPU, so no hosted LLM API is used for answer generation.

For each question, the notebook sends the question and four answer options to the local model, extracts the selected answer letter, submits the answer to the game API, and records the final score.

In the leaderboard our best run reached **1M** in news and $16,000 in Maths. However, the saved notebook output may show a lower score because each rerun starts a new quiz session and the questions can change. Since the model is not always deterministic across different questions and sessions, the final score may vary between runs. For this reason, we report both the saved notebook run and the best leaderboard result.

### How this notebook is organized

This notebook covers the two remaining categories, **News** and **Maths**, and follows the same pipeline as our main notebook:

1. Installing the libraries
2. Connecting to the game and logging in
3. Selecting the categories (News and Maths)
4. Loading the local model (Qwen2.5-7B in 4-bit)
5. The answer strategy (a single greedy answer)
6. Running the quiz
7. Findings and future work

For these two categories we deliberately used a **simpler single-pass strategy** than the voting ensemble in our four-category notebook, so that we could compare a lightweight approach against the more complex one.

In [ ]:
!pip install --quiet -U transformers accelerate "bitsandbytes>=0.46.1" sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 29.1 MB/s eta 0:00:00


This cell installs the libraries needed to download and run the local Qwen model in Google Colab.  
`accelerate` and `bitsandbytes` help load the model efficiently on the GPU using 4-bit quantization.

In [ ]:
!pip install --quiet \
    transformers \
    sentence-transformers \
    torch \
    matplotlib \
    pandas \
    seaborn



The two cells above install everything the notebook needs. The first installs the packages to download and run the local Qwen model on the GPU (`transformers`, `accelerate`, and `bitsandbytes` for 4-bit quantization). The second installs the supporting libraries we use for text processing and for inspecting our results.

## 2. Connecting to the Game and Logging In

In [ ]:
import sys
import zipfile
import os
import random
import time
import gc
import re
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from google.colab import drive

drive.mount("/content/drive")

zip_path = "/content/drive/MyDrive/NLP_assignment_api_client.zip"
extract_path = "/content/nlp_client"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

sys.path.append("/content/nlp_client/NLP_assignment_api_client")

from millionaire_client.client import MillionaireClient

username = "11033318"
password = "Sakthi@2"
base_url = "http://131.175.15.22:51111"

client = MillionaireClient(base_url)
client.login(username, password)

print("Login successful.")

Mounted at /content/drive
Login successful.


This cell connects the notebook to the PoliMillionaire server. We mount Google Drive, unzip the provided API client, add it to the Python path, import `MillionaireClient`, create the client pointing at the game server, and log in with our team account. After this cell, we can list competitions, receive questions, submit answers, and read scores through the text-based API.

In [ ]:
## 3. Selecting the Categories (News and Maths)

In [ ]:
competitions = client.competitions.list_all()

competition_name_by_id = {comp.id: comp.name for comp in competitions}

id_news = next(comp.id for comp in competitions if "News" in comp.name)
id_maths = next(comp.id for comp in competitions if "Math" in comp.name or "Maths" in comp.name)

competition_ids = [
    id_news,
    id_maths
]

categories = {
    id_news: "News",
    id_maths: "Maths"
}

Here we fetch all available competitions and select the two categories for this notebook, **News** and **Maths**, by matching their names. We keep their IDs and readable names in the `categories` and `competition_name_by_id` dictionaries, so that later we start the correct category and label our results clearly.

In [ ]:
## 4. Loading the Local Model (Qwen2.5-7B, 4-bit)

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

login()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
else:
    raise RuntimeError("GPU is not enabled. Go to Runtime > Change runtime type > T4 GPU.")

model_id = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

This cell logs in to Hugging Face, checks that a GPU is available, and loads **Qwen2.5-7B-Instruct** locally with **4-bit quantization** (`bitsandbytes`). Quantization lets a 7-billion-parameter model fit on a single free Colab T4 GPU.

## 5. The Answer Strategy

In [ ]:
import re
import random
import torch

def answer_strategy(question, category_id):
    opts = {chr(65+i): opt for i, opt in enumerate(question.options[:4])}
    options_text = "\n".join([f"{k}. {v.text}" for k, v in opts.items()])

    prompt = f"""
Category:
{categories.get(category_id, "General Knowledge")}

Question:
{question.text}

Options:
{options_text}

Choose the best answer.
Return only A, B, C, or D.
"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).upper()
    match = re.search(r"\b([A-D])\b", raw)
    letter = match.group(1) if match else random.choice(["A", "B", "C", "D"])

    final_option = opts[letter]
    return final_option.id, final_option.text, letter, [letter]

This cell defines a simple answer strategy for the quiz. It takes the question and four answer options, formats them into a prompt, and sends the prompt to the locally loaded Qwen model.

The model returns an answer letter from A to D. The code extracts that letter, matches it to the correct option ID, and returns it so the game API can submit the selected answer.

## 6. Running the Quiz

In [ ]:
final_quest_results = {}
run_history = []

for cid in competition_ids:
    category_name = competition_name_by_id.get(cid, f"Competition {cid}")

    print("\n==============================")
    print(f"STARTING QUEST: {category_name}")
    print("==============================")

    session = client.game.start(competition_id=cid)

    while session.in_progress:
        question = session.current_question

        if not question:
            print("No active question found.")
            break

        level_before = session.current_level

        print(f"\nLevel: {level_before}")
        print("Question:", question.text)

        for i, opt in enumerate(question.options[:4]):
            print(f"{chr(65+i)}. {opt.text}")

        option_id, answer_text, letter, votes = answer_strategy(question, cid)

        try:
            result = session.answer(option_id)

            correct = getattr(result, "correct", None)
            game_over = getattr(result, "game_over", session.is_game_over)
            earned_amount = getattr(result, "earned_amount", session.earned_amount)

            run_history.append({
                "category": category_name,
                "category_id": cid,
                "level": level_before,
                "question": question.text,
                "answer_letter": letter,
                "answer_text": answer_text,
                "votes": votes,
                "correct": correct,
                "earned_amount": earned_amount,
                "game_over": game_over
            })

            print(f"Submitted: {letter} -> {answer_text}")
            print(f"Correct: {correct}")
            print(f"Earned amount now: ${earned_amount}")
            print(f"Game over: {game_over}")

            if game_over:
                break

        except Exception as e:
            print("Submit failed:", e)
            break

        time.sleep(3)

    final_quest_results[category_name] = session.earned_amount
    print(f"\nFinal score for {category_name}: ${session.earned_amount}")

print("\n==============================")
print("FINAL SUMMARY")
print("==============================")

total_earned = sum(final_quest_results.values())

for category, amount in final_quest_results.items():
    print(f"{category}: ${amount}")

print(f"Total earnings: ${total_earned}")


STARTING QUEST: News

Level: 1
Question: On 2026-05-13, the author mentions that the Star Fox 64 remake retains which aspect from the original game?
A. Visual styles
B. Level layouts
C. Character designs
D. Controller rumble technology
Submitted: B -> Level layouts
Correct: True
Earned amount now: $100
Game over: False

Level: 2
Question: According to the news article published on 2026-05-10, what is the name given to the five activists on trial for attacking the Elbit Systems facility in Ulm?
A. The Berlin Five
B. The Ulm 5
C. The Munich Five
D. The Stuttgart Six
Submitted: B -> The Ulm 5
Correct: True
Earned amount now: $200
Game over: False

Level: 3
Question: According to the article published on 2026-05-17, what strategic city is Ukraine's 93rd brigade defending?
A. Kramatorsk
B. Sloviansk
C. Kostyantynivka
D. Donbas
Submitted: C -> Kostyantynivka
Correct: True
Earned amount now: $300
Game over: False

Level: 4
Question: As reported on 2026-05-17, which two Saudi scholars are fac

This cell plays the News and Maths quizzes. For each category it starts a session, loops through the questions, prints each question and its options, calls `answer_strategy`, submits the answer, and records the outcome (level, question, chosen letter, correctness, earnings) in `run_history`. We wait a few seconds (`time.sleep(3)`) between questions to avoid rapid consecutive API requests, as the assignment asks, and we print a final summary at the end.

In this saved run, **News** reached   **64,000** (level 12) before missing a question that referred to a specific dated report (2026-05-15), and Maths reached $100 — correct on the first question but missing an algebra equation at level 2. As noted at the top of the notebook, because each rerun starts a new session with different questions, scores vary between runs; our best leaderboard results were higher.

## 7. Findings and Future Work

These two categories show clearly where a plain language model reaches its limits:

- **News is time-sensitive.** Our News failure was a question about a specific dated report. The model cannot know fresh, dated facts from its training data, so this is the clearest case for **Retrieval-Augmented Generation (RAG)**: retrieving raw content from a free source (for example DuckDuckGo or a newspaper API), restricted to articles published before the question date, and giving it to the model as context. *This was not implemented in the current notebook.*
- **Maths needs exact computation.** Our Maths failure was an algebra equation. Language models are not reliable calculators, so a **calculator / SymPy tool** that detects equation-style questions and solves them exactly would be the most useful addition here. *This was not implemented in the current notebook.*

Comparing the two notebooks, the voting ensemble used for the first four categories was more robust than the single greedy answer used here, which supports our choice of the ensemble as our main strategy.